# Verify MVTec Dataset, Checkpoint, and Inference

Loads images, folder/class metadata, ADVIS VAE-GAN checkpoint, reconstruction preview, threshold calibration, and subset inference.

In [ ]:
from pathlib import Path
import sys

NOTEBOOK_DIR = Path.cwd() if Path.cwd().name == "notebooks" else Path.cwd() / "notebooks"
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

from verification_helpers import *

CONFIG_NAME = "mvtec.yaml"
MAX_TRAIN_SAMPLES = 512
MAX_TEST_SAMPLES = 512

In [ ]:
state = load_config_data(CONFIG_NAME)
cfg = state["cfg"]
project_root = state["project_root"]
device = state["device"]
train_dataset = state["train_dataset"]
test_dataset = state["test_dataset"]

print("config:", state["config_path"])
print("device:", device)
print("dataset:", cfg.data.name, cfg.data.category)
print("train:", state["summary"]["train"]["class_counts"])
print("test:", state["summary"]["test"]["class_counts"])
state["sample_records"](test_dataset, 5)

In [ ]:
display(show_samples(train_dataset, "MVTec train images", max_images=6))
display(show_samples(test_dataset, "MVTec test images", max_images=8))

In [ ]:
checkpoint_path, encoder, decoder, discriminator = load_vaegan_model(cfg, device)
print("checkpoint:", checkpoint_path)
print("exists:", checkpoint_path.exists())
show_reconstructions(encoder, decoder, state["test_loader"], device, max_images=6)

In [ ]:
training_history, training_artifacts = show_training_curves(project_root, cfg, checkpoint_path)
training_artifacts


In [ ]:
metrics, results_df, threshold_model = run_subset_inference(
    cfg, encoder, decoder, discriminator,
    train_dataset, test_dataset,
    max_train_samples=MAX_TRAIN_SAMPLES,
    max_test_samples=MAX_TEST_SAMPLES,
)
display(metrics)
display(results_df.head())
display(results_df.groupby("folder_name")[["binary_label", "prediction"]].agg(["count", "mean"]))

In [ ]:
output_dir = save_notebook_outputs(project_root, cfg.data.name, cfg.data.category, metrics, results_df, state["summary"])
print(output_dir)